# Notebook 18 — frozen ViT mathematical diagnostics

This definition notebook supplies the short TRAIN-only diagnostic stage to the full study. It measures query/key magnitudes, exponential-feature concentration, exact attention outside the original 3×3 patch neighborhood, and singular spectra of actual geometrically separated attention blocks. Three independent feature bases at ranks 32, 64 and 128 compare raw and mathematically compensated centering. It does not measure classification, prove approximation error bounds, or select a favorable basis for the learned model.

Inputs: completed frozen Notebook 15 ViT seed 20261701, four preregistered TRAIN image IDs sampled with seed 2026181801, full-image EMA target activations at first and last blocks, tagged Notebook 17 basis definitions, and the source-bound Notebook 18 helper. Official test images are never read. Outputs: diagnostic JSON/case logs, source-bound captures and RNG/basis state. Expected runtime: several minutes with a hard 15-minute phase budget; every case has a hard 120-second subprocess limit.

Checkpoint location: `checkpoints/<run_id>/diagnostic/`, latest two valid states plus best numerical approximation state. For the full study, the coordinator passes a uniquely named diagnostic directory. Resume exactly: Run All in `notebooks/18_Learned_Kernel_JEPA_Diagnostic_and_Learning.ipynb` using the same `FMM18_RUN_ID`; the `launch-learned-kernel-study` cell restores the original stage clock and request. Invalid hashes/source or orphan state raise an error. An interrupted case stays UNKNOWN and is not silently retried. There is one writer per stage.

The replacement being trained has its own learned positive kernel; failed frozen-ViT approximation diagnostics do not block its independent learning experiment. The frozen source weights are immutable, so optimizer/scheduler/scaler are explicitly not applicable here. Captured QKV/projection weights, image IDs, basis matrices, metrics, position and every initialized RNG are checkpointed.


### Why key compensation is required

For a common differentiable center $c$, write $q'=q-c$ and $k'=k-c$. Then

$$q^	op k/	au=q'^	op k'/	au+c^	op k'/	au+c^	op q'/	au+\|c\|^2/	au.$$

The last two terms are constant across keys for each query and cancel in row softmax. The term $c^	op k'/	au$ depends on the key and must remain. Positive random features use this log weight before common stabilizing shifts. The exact identity is independently checked for outputs and all input/temperature derivatives.

For an IID Gaussian feature direction, the squared coefficient of variation of a single positive kernel estimate is $\exp(\|q+k\|^2/	au)-1$, obtained directly from the Gaussian moment-generating function. After common translation it becomes $\exp(\|q+k-2c\|^2/	au)-1$; the deterministic key compensation cancels in this relative variance. Our orthogonal feature rows are dependent, so these quantities diagnose risk and are not exact orthogonal-basis variances. Row normalization introduces further nonlinear effects.


In [ ]:
from pathlib import Path
import contextlib, copy, fcntl, gc, hashlib, json, math, os, random, subprocess, sys, time, types
import numpy as np
import torch
from torch import nn
ROOT=Path(os.environ.get('FMM_JEPA_ROOT','/home/benjamin/workspace/ahr_go/fmm_jeppa')).resolve()
if '__file__' not in globals():__file__=str(ROOT/'experiments/18_learned_kernel/diagnostics.ipynb')
_CACHE={}
SCHEMA='fmm18-train-only-diagnostic/v1'
BASIS_SEEDS=(180032,180064,180128)
RANKS=(32,64,128)


def load(root,relative,name):
    path=(Path(root)/relative).resolve();key=(str(path),name)
    if key not in _CACHE:
        module=types.ModuleType(name);module.__file__=str(path)
        for cell in json.loads(path.read_text())['cells']:
            if cell['cell_type']=='code'and'ahr-export'in cell.get('metadata',{}).get('tags',[]):
                exec(compile(''.join(cell['source']),str(path)+'#ahr-export','exec'),module.__dict__)
        _CACHE[key]=module
    return _CACHE[key]


def operator(root=ROOT):return load(root,'experiments/17_fixed_kernel/operator.ipynb','fmm18_diagnostic_operator17')
def common17(root=ROOT):return load(root,'experiments/17_fixed_kernel/common.ipynb','fmm18_diagnostic_common17')
def common18(root=ROOT):return load(root,'experiments/18_learned_kernel/common.ipynb','fmm18_diagnostic_common18')
def canonical(value):return json.loads(json.dumps(value,sort_keys=True,allow_nan=False))
def digest(value):return hashlib.sha256(json.dumps(canonical(value),sort_keys=True,separators=(',',':')).encode()).hexdigest()
def read(path):return json.loads(Path(path).read_text())
def file_record(path):
    path=Path(path).resolve();h=hashlib.sha256()
    with path.open('rb')as handle:
        for block in iter(lambda:handle.read(2**20),b''):h.update(block)
    return {'path':str(path),'sha256':h.hexdigest(),'bytes':path.stat().st_size}
def atomic_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);temporary=path.with_name(path.name+'.'+str(os.getpid())+'.tmp')
    with temporary.open('w')as handle:json.dump(canonical(value),handle,sort_keys=True,indent=2);handle.flush();os.fsync(handle.fileno())
    os.replace(temporary,path)
def atomic_torch(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);temporary=path.with_name(path.name+'.'+str(os.getpid())+'.tmp')
    with temporary.open('wb')as handle:torch.save(value,handle);handle.flush();os.fsync(handle.fileno())
    os.replace(temporary,path)
def immutable(path,value):
    path=Path(path);value=canonical(value)
    if path.exists():
        if read(path)!=value:raise ValueError('Immutable diagnostic request/source changed; never restart: '+str(path))
    else:atomic_json(path,value)
    return value


def source_hashes(root=ROOT,profile='full'):
    root=Path(root)
    if profile=='full':
        path=root/'experiments/18_learned_kernel/common.ipynb'
        if not path.exists():raise FileNotFoundError('Source-bound common18 export required before scientific launch')
        return common18(root).source_hashes(root)
    path=Path(__file__);nb=json.loads(path.read_text())
    exported='\n'.join(''.join(c['source'])for c in nb['cells']if c['cell_type']=='code'and'ahr-export'in c.get('metadata',{}).get('tags',[]))
    return {'diagnostic_export':hashlib.sha256(exported.encode()).hexdigest(),'operator17_export':common17(root).export_hash(root/'experiments/17_fixed_kernel/operator.ipynb')}


def check_deadline(deadline,required=0.):
    if deadline is not None and time.time()+float(required)>=float(deadline):raise TimeoutError('Original diagnostic wall budget exhausted; UNKNOWN')


def rng_state():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch_cpu':torch.get_rng_state().clone(),
      'torch_cuda_all':torch.cuda.get_rng_state_all()if torch.cuda.is_initialized()else None}
def restore_rng(value):
    random.setstate(value['python']);np.random.set_state(value['numpy']);torch.set_rng_state(value['torch_cpu'])
    if value['torch_cuda_all']is not None:torch.cuda.set_rng_state_all(value['torch_cuda_all'])


def positive_temperature(tau,q):
    result=torch.as_tensor(tau,device=q.device,dtype=q.dtype).reshape(q.shape[1])
    if not bool(torch.isfinite(result).all())or bool((result<=0).any()):raise ValueError('Finite positive temperature per head required')
    return result


def compensated_center(q,k,tau):
    """A shared, differentiable translation with required key compensation.

    q=q'+c and k=k'+c imply q.k/tau=q'.k'/tau+c.k'/tau
    +c.q'/tau+||c||^2/tau. The last two terms are query-only and cancel
    in softmax. The key term MUST remain; its omission changes attention.
    """
    temperature=positive_temperature(tau,q)
    center=.5*(q.mean(-2,keepdim=True)+k.mean(-2,keepdim=True))
    qc,kc=q-center,k-center
    key_compensation=(kc*center).sum(-1)/temperature[None,:,None]
    return qc,kc,key_compensation,center


def exact_attention(q,k,v,tau,centered=False,return_weights=False):
    temperature=positive_temperature(tau,q)
    if centered:
        qc,kc,beta,_=compensated_center(q,k,temperature)
        logits=(qc@kc.transpose(-1,-2))/temperature[None,:,None,None]+beta[:,:,None]
    else:logits=(q@k.transpose(-1,-2))/temperature[None,:,None,None]
    weights=logits.softmax(-1);output=weights@v
    return(output,weights)if return_weights else output


def local_mask(indices,grid,radius=1):
    h,w=map(int,grid)
    if indices.ndim!=2 or indices.dtype!=torch.long or bool(((indices<0)|(indices>=h*w)).any()):raise ValueError('Unpadded original patch IDs required')
    row,col=indices//w,indices%w
    return((row[:,:,None]-row[:,None,:]).abs()<=radius)&((col[:,:,None]-col[:,None,:]).abs()<=radius)


def log_features(q,k,W,tau,centered=False):
    temperature=positive_temperature(tau,q)
    if centered:qc,kc,beta,center=compensated_center(q,k,temperature)
    else:qc,kc,beta,center=q,k,torch.zeros_like(k[...,0]),torch.zeros_like(q[...,:1,:])
    scale=temperature.sqrt()[None,:,None,None];x,y=qc/scale,kc/scale
    logq=torch.einsum('bhnd,hrd->bhnr',x,W)-.5*x.square().sum(-1,keepdim=True)
    logk=torch.einsum('bhnd,hrd->bhnr',y,W)-.5*y.square().sum(-1,keepdim=True)+beta[...,None]
    exact_scores=(qc@kc.transpose(-1,-2))/temperature[None,:,None,None]+beta[:,:,None]
    return logq,logk,exact_scores,center


def finite_attention(q,k,v,W,tau,indices,grid,centered=False,return_details=False):
    """Dense diagnostic reference at N<=256, never the scalable encoder.

    One common key shift per sample/head and one query shift per row cancel
    from normalization. Per-key compensation is applied BEFORE those shifts.
    Exact local weights use the same translated kernel scale. All far weights
    are positive finite features; no low-rank or speed claim follows.
    """
    if q.shape[-2]>256:raise ValueError('Bounded dense diagnostic refuses N>256')
    logq,logk,scores,center=log_features(q,k,W,tau,centered)
    near=local_mask(indices,grid)
    key_shift=logk.amax((-2,-1)).detach()
    query_shift=torch.maximum(logq.amax(-1),scores.masked_fill(~near[:,None],-torch.inf).amax(-1)-key_shift[:,:,None]).detach()
    phi_q=torch.exp(logq-query_shift[...,None])/math.sqrt(W.shape[1])
    phi_k=torch.exp(logk-key_shift[:,:,None,None])/math.sqrt(W.shape[1])
    approximate=phi_q@phi_k.transpose(-1,-2)
    local=torch.exp(scores.masked_fill(~near[:,None],-torch.inf)-query_shift[...,None]-key_shift[:,:,None,None])
    weights=torch.where(near[:,None],local,approximate);denominator=weights.sum(-1,keepdim=True)
    if not bool(torch.isfinite(denominator).all())or bool((denominator<=torch.finfo(q.dtype).tiny).any()):raise FloatingPointError('Finite-feature denominator underflow/nonpositive; UNKNOWN')
    result=(weights@v)/denominator
    if not bool(torch.isfinite(result).all()):raise FloatingPointError('Nonfinite approximation; UNKNOWN')
    return(result,{'logq':logq,'logk':logk,'center':center,'phi_q':phi_q,'phi_k':phi_k})if return_details else result


def relative(left,right):
    denominator=right.detach().double().norm();numerator=(left.detach()-right.detach()).double().norm()
    if float(denominator)==0:return 0. if float(numerator)==0 else None
    value=float(numerator/denominator)
    if not math.isfinite(value):raise FloatingPointError('Nonfinite relative numerical error; UNKNOWN')
    return value


def quantiles(value):
    flat=value.detach().double().reshape(-1).cpu()
    if not bool(torch.isfinite(flat).all()):raise FloatingPointError('Nonfinite measured distribution')
    levels=(0.,.25,.5,.75,.9,.99,1.)
    if flat.numel()==0:return {'count':0,'quantile_levels':list(levels),'values':[None]*len(levels),'min':None,'median':None,'p99':None,'max':None,'mean':None}
    measured=torch.quantile(flat,torch.tensor(levels,dtype=torch.float64))
    return {'count':int(flat.numel()),'quantile_levels':list(levels),'values':measured.tolist(),
      'min':float(measured[0]),'median':float(measured[2]),'p99':float(measured[-2]),'max':float(measured[-1]),'mean':float(flat.mean())}


def feature_concentration(details):
    result={}
    for side in ('q','k'):
        log=details['log'+side].detach();normalized=log.softmax(-1)
        effective=normalized.square().sum(-1).reciprocal()
        result[side]={'effective_feature_count':quantiles(effective),'largest_feature_share':quantiles(normalized.amax(-1)),
          'log_feature_range':quantiles(log.amax(-1)-log.amin(-1)),
          'floating_zero_fraction_after_matched_scaling':float((details['phi_'+side].detach()==0).double().mean())}
    return result


def token_statistics(q,k,tau,indices,grid):
    temperature=positive_temperature(tau,q)
    _,attention=exact_attention(q,k,k,tau,return_weights=True)
    near=local_mask(indices,grid)
    outside=(attention*~near[:,None]).sum(-1)
    combined=q.square().sum(-1)[...,None]+k.square().sum(-1)[:,:,None,:]+2*(q@k.transpose(-1,-2))
    variance_indicator=combined.clamp_min(0)/temperature[None,:,None,None]
    qc,kc,_,center=compensated_center(q,k,tau)
    shifted=qc.square().sum(-1)[...,None]+kc.square().sum(-1)[:,:,None,:]+2*(qc@kc.transpose(-1,-2))
    return {'query_norm':quantiles(q.norm(dim=-1)),'key_norm':quantiles(k.norm(dim=-1)),
      'query_norm_over_sqrt_tau':quantiles(q.norm(dim=-1)/temperature.sqrt()[None,:,None]),
      'key_norm_over_sqrt_tau':quantiles(k.norm(dim=-1)/temperature.sqrt()[None,:,None]),
      'pair_sum_squared_over_tau':quantiles(variance_indicator),
      'far_pair_sum_squared_over_tau':quantiles(variance_indicator.masked_select((~near[:,None]).expand_as(variance_indicator))),
      'centered_pair_sum_squared_over_tau':quantiles(shifted.clamp_min(0)/temperature[None,:,None,None]),
      'centered_far_pair_sum_squared_over_tau':quantiles((shifted.clamp_min(0)/temperature[None,:,None,None]).masked_select((~near[:,None]).expand_as(shifted))),
      'center_norm':quantiles(center.norm(dim=-1)),'exact_attention_mass_outside_3x3':quantiles(outside),
      'variance_indicator_scope':'For IID Gaussian positive features, relative variance of one kernel estimate is exp(||q+k||²/tau)-1. Orthogonal features are dependent: this measured quantity is a variance-risk indicator, not an exact ORF variance or causal diagnosis.'}


def far_block_spectra(q,k,tau,indices,grid):
    """True separated spatial blocks; no arbitrary random submatrix rank proxy."""
    h,w=map(int,grid);row,col=indices[0]//w,indices[0]%w
    if h<8 or w<8:return []
    width=max(1,h//4);cols=max(1,w//4)
    selections=[('top_to_bottom',row<width,row>=h-width),('bottom_to_top',row>=h-width,row<width),
      ('left_to_right',col<cols,col>=w-cols),('right_to_left',col>=w-cols,col<cols)]
    _,attention=exact_attention(q,k,k,tau,return_weights=True);near=local_mask(indices,grid);result=[]
    for name,query_selection,key_selection in selections:
        query_indices=torch.where(query_selection)[0];key_indices=torch.where(key_selection)[0]
        if not len(query_indices)or not len(key_indices):continue
        if bool(near[:,query_indices][:,:,key_indices].any()):raise AssertionError('Diagnostic far block contains a local pair')
        block=attention[:,:,query_indices][:,:,:,key_indices].double();singular=torch.linalg.svdvals(block)
        energy=singular.square();total=energy.sum(-1,keepdim=True)
        if bool((total<=0).any()):raise FloatingPointError('Zero far block energy; rank UNKNOWN')
        cumulative=energy.cumsum(-1)/total;probability=energy/total
        effective=torch.exp(-(probability*probability.clamp_min(torch.finfo(torch.float64).tiny).log()).sum(-1))
        ranks10=(cumulative>=.99).long().argmax(-1)+1;ranks1=(cumulative>=.9999).long().argmax(-1)+1
        result.append({'name':name,'shape':list(block.shape),'query_original_ids':indices[0,query_indices].tolist(),'key_original_ids':indices[0,key_indices].tolist(),
          'all_pairs_outside_3x3':True,'singular_values':singular.tolist(),'relative_singular_values':(singular/singular[...,:1]).tolist(),
          'rank_for_10_percent_frobenius_error':ranks10.tolist(),'rank_for_1_percent_frobenius_error':ranks1.tolist(),
          'energy_effective_rank':effective.tolist(),'matrix_role':'Exact row-normalized trained ViT attention restricted to genuine geometrically separated blocks; empirical compressibility only'})
    return result


def equivalence_certificate(q,k,v,tau,weights=None):
    values=[value.detach().double().clone().requires_grad_()for value in(q,k,v)]
    temp=torch.as_tensor(tau,dtype=torch.float64).detach().clone().requires_grad_()
    raw=exact_attention(*values,temp);centered=exact_attention(*values,temp,centered=True)
    objective=torch.arange(raw.numel(),dtype=torch.float64).reshape_as(raw).sin()if weights is None else weights.double()
    raw_grad=torch.autograd.grad((raw*objective).sum(),values+[temp],retain_graph=True)
    centered_grad=torch.autograd.grad((centered*objective).sum(),values+[temp])
    errors={name:relative(left,right)for name,left,right in zip(('query','key','value','temperature'),centered_grad,raw_grad)}
    output=relative(centered,raw);passed=output is not None and output<1e-10 and all(error is not None and error<1e-9 for error in errors.values())
    return {'status':'PASS'if passed else'FAIL','passed':passed,'relative_output_l2':output,'relative_gradient_l2':errors,
      'arithmetic':'FP64 independent exact logits; center remains differentiable and key compensation included'}


def _projection(mixer_state):
    width=int(mixer_state['width']);qkv=nn.Linear(width,3*width,bias=mixer_state['qkv_bias']is not None)
    projection=nn.Linear(width,width,bias=mixer_state['projection_bias']is not None)
    qkv.weight.data.copy_(mixer_state['qkv_weight']);projection.weight.data.copy_(mixer_state['projection_weight'])
    if qkv.bias is not None:qkv.bias.data.copy_(mixer_state['qkv_bias'])
    if projection.bias is not None:projection.bias.data.copy_(mixer_state['projection_bias'])
    return qkv,projection


def analyze_case(root,job):
    torch.set_num_threads(2);torch.use_deterministic_algorithms(True)
    if source_hashes(root,job['profile'])!=job['source_hashes']:raise ValueError('Diagnostic child source changed')
    check_deadline(job['deadline'],1)
    if job.get('capture_only'):
        record,_=_capture(root,Path(job['artifact']),job['profile'],job['deadline'])
        row={'key':job['key'],'status':'CAPTURED','capture_record':record,'source_hashes':job['source_hashes'],'request_sha256':job['request_sha256'],'official_test_pixels_read':0}
        atomic_json(job['result_path'],row);return row
    if file_record(job['capture_record']['path'])!=job['capture_record']:raise ValueError('Frozen capture checksum changed')
    capture=torch.load(job['capture_record']['path'],map_location='cpu',weights_only=False)
    layer=capture['layers'][str(job['layer'])];tokens=layer['tokens'].clone().requires_grad_();qkv,projection=_projection(layer['mixer'])
    B,N,width=tokens.shape;heads=int(layer['mixer']['heads']);D=width//heads
    q,k,v=qkv(tokens).reshape(B,N,3,heads,D).permute(2,0,3,1,4).unbind(0)
    tau=torch.full((heads,),math.sqrt(D),dtype=tokens.dtype,requires_grad=True)
    basis=operator(root).make_orthogonal_features(heads,D,job['rank'],job['feature_seed'],dtype=tokens.dtype)
    indices,grid=layer['indices'],tuple(layer['grid']);output_weights=torch.arange(B*N*width,dtype=tokens.dtype).reshape(B,N,width).sin()
    exact_av=exact_attention(q,k,v,tau);exact_output=projection(exact_av.transpose(1,2).reshape(B,N,width))
    parameters=[tokens,*qkv.parameters(),*projection.parameters(),tau]
    names=['input_tokens','qkv.weight']+(['qkv.bias']if qkv.bias is not None else[])+['projection.weight']+(['projection.bias']if projection.bias is not None else[])+['temperature']
    exact_grads=torch.autograd.grad((exact_output*output_weights).sum(),parameters,retain_graph=True)
    measured={}
    for centered in (False,True):
        check_deadline(job['deadline'],1)
        av,details=finite_attention(q,k,v,basis,tau,indices,grid,centered=centered,return_details=True)
        output=projection(av.transpose(1,2).reshape(B,N,width));gradients=torch.autograd.grad((output*output_weights).sum(),parameters,retain_graph=True)
        errors={name:relative(left,right)for name,left,right in zip(names,gradients,exact_grads)}
        measured['centered'if centered else'raw']={'relative_attention_value_output_l2':relative(av,exact_av),'relative_projected_output_l2':relative(output,exact_output),
          'relative_shared_parameter_gradient_l2':errors,'max_relative_shared_parameter_gradient_l2':max(v for v in errors.values()if v is not None),
          'positive_feature_concentration':feature_concentration(details),'basis_sha256':operator(root).basis_sha256(basis)}
    certificate=equivalence_certificate(q.detach(),k.detach(),v.detach(),tau.detach())
    row={'key':job['key'],'status':'MEASURED','layer':job['layer'],'tokens':N,'rank':job['rank'],'feature_seed':job['feature_seed'],
      'source_hashes':job['source_hashes'],'request_sha256':job['request_sha256'],'sample_ids':capture['sample_ids'],'source':capture['source'],
      'measurements':measured,'centering_exact':certificate,'token_statistics':layer['token_statistics'],'far_blocks':layer['far_blocks'],
      'role':'SYNTHETIC_SOFTWARE_VALIDATION'if job['profile']!='full'else'FROZEN_TRAIN_ONLY_NUMERICAL_DIAGNOSTIC',
      'official_test_pixels_read':0,'no_downstream_accuracy_or_speed_claim':True,'gradient_probe':'One deterministic VJP: projected outputs dotted with sin(ordinal output coordinate); all named shared parameters measured, no uniform Jacobian-error certificate','finished_unix':time.time()}
    atomic_torch(Path(job['result_path']).with_suffix('.rng.pt'),{'rng':rng_state(),'basis':basis,'job':job,'source':capture['source'],'position':job['key']})
    row['state_record']=file_record(Path(job['result_path']).with_suffix('.rng.pt'));atomic_json(job['result_path'],row);return row


def _capture(root,artifact,profile,deadline):
    torch.set_num_threads(2);torch.use_deterministic_algorithms(True);check_deadline(deadline,5)
    if profile!='full':
        generator=torch.Generator().manual_seed(18018);width=12;heads=2;N=16
        tokens=torch.randn(2,N,width,generator=generator)*.25;mixer=nn.Module();mixer.heads=heads;mixer.qkv=nn.Linear(width,3*width);mixer.projection=nn.Linear(width,width)
        source={'role':'SYNTHETIC_SOFTWARE_VALIDATION','encoder':'SYNTHETIC_PROJECTIONS'};sample_ids=[];captured=[(0,tokens,mixer,(4,4))]
        sampler_rng={'kind':'LOCAL_TORCH_SYNTHETIC_GENERATOR','state':generator.get_state()}
    else:
        C=common17(root);model,config,source=C.learned_source(root,variant='vit');config['device']='cpu'
        if int(source['seed'])!=20261701:raise ValueError('Registered first frozen ViT seed changed')
        bundle=common18(root).data_bundle(root,artifact/'source_data',config,verify_images=False)
        rng=np.random.default_rng(2026181801);sample_ids=sorted(rng.choice(bundle['train_ids'],size=4,replace=False).tolist())
        sampler_rng={'kind':'LOCAL_NUMPY_REGISTERED_TRAIN_SAMPLING','state':copy.deepcopy(rng.bit_generator.state),'training_membership_sha256':digest(sorted(bundle['train_ids']))}
        source={**source,'encoder':'FROZEN_EMA_TARGET_FULL_IMAGE_TOKENS'}
        if any(i not in set(bundle['train_ids'])for i in sample_ids):raise AssertionError('Diagnostic sampled outside TRAIN membership')
        from fmm_jepa.food101_data import EvaluationDataset
        changed=copy.deepcopy(config);changed['image_size']=256;data=EvaluationDataset(bundle['dataset'],sample_ids,changed)
        images=torch.stack([data[i]['image']for i in range(len(sample_ids))]);target=model.target.cpu().eval();layers=[0,len(target.blocks)-1];observed={};handles=[]
        for layer in layers:
            handles.append(target.blocks[layer].norm1.register_forward_hook(lambda module,args,out,index=layer:observed.setdefault(index,out.detach().clone())))
        try:
            with torch.no_grad():target(images)
        finally:
            for handle in handles:handle.remove()
        grid=256//int(config['model']['patch_size']);captured=[(layer,observed[layer],copy.deepcopy(target.blocks[layer].mixer).cpu(),(grid,grid))for layer in layers]
        model.cpu();del model;gc.collect()
    capture={'schema':SCHEMA,'source':source,'sample_ids':sample_ids,'layers':{},'official_test_pixels_read':0}
    for layer,tokens,mixer,grid in captured:
        check_deadline(deadline,1);B,N,width=tokens.shape;heads=int(mixer.heads);D=width//heads;indices=torch.arange(N).expand(B,-1).clone()
        with torch.no_grad():q,k,v=mixer.qkv(tokens).reshape(B,N,3,heads,D).permute(2,0,3,1,4).unbind(0)
        tau=torch.full((heads,),math.sqrt(D),dtype=tokens.dtype)
        record={'width':width,'heads':heads,'qkv_weight':mixer.qkv.weight.detach().clone(),'qkv_bias':mixer.qkv.bias.detach().clone()if mixer.qkv.bias is not None else None,
          'projection_weight':mixer.projection.weight.detach().clone(),'projection_bias':mixer.projection.bias.detach().clone()if mixer.projection.bias is not None else None}
        capture['layers'][str(layer)]={'tokens':tokens,'mixer':record,'indices':indices,'grid':grid,'token_statistics':token_statistics(q,k,tau,indices,grid),'far_blocks':far_block_spectra(q,k,tau,indices,grid)}
    capture['rng']=rng_state();capture['sampling_rng']=sampler_rng;path=artifact/'frozen_train_capture.pt';atomic_torch(path,capture);return file_record(path),capture


def save_checkpoint(folder,identity,completed,capture_record=None,active_case=None):
    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);manifest_path=folder/'manifest.json';previous=read(manifest_path)if manifest_path.exists()else None
    if previous and previous['identity']!=identity:raise ValueError('Diagnostic checkpoint identity changed')
    sequence=int(previous['sequence'])+1 if previous else 1;path=folder/('diagnostic-'+str(sequence).zfill(6)+'.pt')
    payload={'schema':SCHEMA,'identity':identity,'completed':completed,'position':len(completed),'active_case':active_case,'capture_record':capture_record,
      'rng':rng_state(),'model_state':'Immutable frozen source checkpoint + checksummed captured tokens and actual QKV/projection weights; never trained',
      'optimizer':None,'scheduler':None,'scaler':None,'sampler_state':{'registered_training_ids_in_capture':capture_record,'completed_case_keys':sorted(completed),'next_in_flight':active_case},
      'config':identity,'metrics':completed,'sequence':sequence}
    atomic_torch(path,payload);record=file_record(path)
    measured=[row for row in completed.values()if row.get('status')=='MEASURED'];score=min((row['measurements']['centered']['relative_attention_value_output_l2']for row in measured),default=None)
    improve=previous is None or(score is not None and(previous.get('best_score')is None or score<previous['best_score']))
    best=record if improve else previous['best'];latest_two=([record]+(previous['latest_two']if previous else[]))[:2]
    manifest={'schema':SCHEMA,'identity':identity,'sequence':sequence,'latest':record,'latest_two':latest_two,'best':best,'best_score':score if improve else previous.get('best_score'),
      'completed_count':len(completed),'active_case':active_case,'single_writer_pid':os.getpid(),'resume_notebook':'notebooks/18_Learned_Kernel_JEPA_Diagnostic_and_Learning.ipynb','resume_cell':'launch-learned-kernel-study'}
    atomic_json(manifest_path,manifest);keep={r['path']for r in latest_two+[best]}
    for candidate in folder.glob('diagnostic-*.pt'):
        if str(candidate.resolve())not in keep:candidate.unlink()
    return manifest


def restore_checkpoint(folder,identity):
    folder=Path(folder);path=folder/'manifest.json'
    if not path.exists():
        if folder.exists()and any(folder.glob('diagnostic-*.pt')):raise ValueError('Orphan diagnostic checkpoint; never silently restart')
        return {},None,None
    manifest=read(path)
    if manifest['identity']!=identity:raise ValueError('Diagnostic source/request/clock mismatch')
    for record in manifest['latest_two']+[manifest['best']]:
        if file_record(record['path'])!=record:raise ValueError('Diagnostic checkpoint hash mismatch')
    payload=torch.load(manifest['latest']['path'],map_location='cpu',weights_only=False)
    if payload['identity']!=identity or payload['position']!=len(payload['completed'])or payload['position']!=manifest['completed_count']:raise ValueError('Diagnostic checkpoint position/metrics mismatch')
    if payload['capture_record']and file_record(payload['capture_record']['path'])!=payload['capture_record']:raise ValueError('Frozen TRAIN capture hash mismatch')
    for row in payload['completed'].values():
        if row.get('state_record')and file_record(row['state_record']['path'])!=row['state_record']:raise ValueError('Diagnostic basis/RNG state mismatch')
    restore_rng(payload['rng']);return payload['completed'],payload['capture_record'],payload['active_case']


def _subprocess_case(root,job_path,timeout):
    root=Path(root);job=read(job_path);code="import json,sys; from pathlib import Path; p=Path(sys.argv[1]); namespace={'__file__':str(p)}; [exec(compile(''.join(c['source']),str(p)+'#ahr-export','exec'),namespace)for c in json.loads(p.read_text())['cells']if c['cell_type']=='code'and'ahr-export'in c.get('metadata',{}).get('tags',[])]; namespace['analyze_case'](Path(sys.argv[2]),namespace['read'](sys.argv[3]))"
    log_path=Path(job['result_path']).with_suffix('.log');log_path.parent.mkdir(parents=True,exist_ok=True)
    with log_path.open('w')as handle:
        process=subprocess.Popen([sys.executable,'-c',code,str(root/'experiments/18_learned_kernel/diagnostics.ipynb'),str(root),str(job_path)],stdout=handle,stderr=subprocess.STDOUT)
        try:process.wait(timeout=max(.01,float(timeout)))
        except subprocess.TimeoutExpired:
            process.kill();process.wait();return {'key':job['key'],'status':'UNKNOWN_CASE_TIMEOUT','reason':'Hard case limit reached; no retry or renewed phase clock','log':str(log_path),'finished_unix':time.time()}
    if process.returncode!=0:return {'key':job['key'],'status':'UNKNOWN_CASE_ERROR','reason':'Fresh diagnostic subprocess failed; inspect retained log','exit_code':process.returncode,'log':str(log_path),'finished_unix':time.time()}
    row=read(job['result_path'])
    if row.get('source_hashes')!=job['source_hashes']or row.get('request_sha256')!=job['request_sha256']:raise ValueError('Diagnostic result source/request mismatch')
    return row


def summarize(rows):
    measured=[row for row in rows if row.get('status')=='MEASURED'];raw=[row['measurements']['raw']['relative_attention_value_output_l2']for row in measured];centered=[row['measurements']['centered']['relative_attention_value_output_l2']for row in measured]
    return {'measured_cases':len(measured),'unknown_cases':len(rows)-len(measured),'centering_exact_equivalence_all_measured_pass':all(row['centering_exact']['passed']for row in measured)if measured else None,
      'raw_median_relative_attention_value_error':float(np.median(raw))if raw else None,'centered_median_relative_attention_value_error':float(np.median(centered))if centered else None,
      'centered_better_count':sum(b<a for a,b in zip(raw,centered)),'rank_and_feature_seed_sweep':'Independent bases; no best seed or rank selected for learning',
      'interpretation':'Small frozen TRAIN-only diagnostics identify measured patterns. They do not prove causes, a uniform approximation bound, complete JEPA speed, or classification superiority.',
      'learning_blocked_by_fidelity':False,'official_test_pixels_read':0}


def _run_diagnostic(root,artifact,deadline=None,profile='full'):
    root,artifact=Path(root).resolve(),Path(artifact).resolve();artifact.mkdir(parents=True,exist_ok=True)
    if profile not in('full','validation'):raise ValueError('Registered profile must be full or validation')
    artifact.relative_to(root/'artifacts');run_id=artifact.parent.name if artifact.name=='diagnostic' else artifact.name
    if not run_id or run_id in('.', '..'):raise ValueError('Unique diagnostic run ID required')
    sources=source_hashes(root,profile);request={'schema':SCHEMA,'profile':profile,'source_hashes':sources,'run_id':run_id,'external_deadline_unix':deadline,
      'wall_budget_seconds':900 if profile=='full'else 120,'case_timeout_seconds':120,'ranks':list(RANKS)if profile=='full'else[8],
      'basis_seeds':list(BASIS_SEEDS)if profile=='full'else[180032],'training_sampling_seed':2026181801,'training_image_count':4 if profile=='full'else 0,
      'tokens':256 if profile=='full'else 16,'layers':'first and last frozen15 ViT EMA target blocks on full images'if profile=='full'else'one synthetic layer',
      'frozen_seed':20261701 if profile=='full'else None,'centering':'Shared differentiable c=(mean q + mean k)/2; compensate keys with c·(k−c)/tau',
      'quality_and_speed':'Not measured; diagnostic failures never gate learned encoder quality tests','official_test_pixels_read':0}
    request=immutable(artifact/'diagnostic_registration.json',request);request_sha=digest(request);clock_path=artifact/'diagnostic_clock.json'
    if clock_path.exists():
        clock=read(clock_path)
        if clock['request_sha256']!=request_sha:raise ValueError('Original diagnostic clock/request changed')
    else:
        started=time.time();clock={'request_sha256':request_sha,'started_unix':started,'deadline_unix':min(started+request['wall_budget_seconds'],float(deadline))if deadline is not None else started+request['wall_budget_seconds']}
        immutable(clock_path,clock)
    limit=float(clock['deadline_unix']);identity={'request_sha256':request_sha,'request':request,'clock':clock};folder=root/'checkpoints'/run_id/'diagnostic'
    completed,capture_record,active=restore_checkpoint(folder,identity)
    if active is not None:
        completed.setdefault(active,{'key':active,'status':'UNKNOWN_INTERRUPTED_CASE','reason':'Checkpoint recorded IN_FLIGHT; no unreported time forgiven or retry'})
        save_checkpoint(folder,identity,completed,capture_record,None)
    try:
        if capture_record is None and 'FROZEN_TRAIN_CAPTURE' in completed:
            report={'schema':SCHEMA,'status':'UNKNOWN','profile':profile,'request':request,'clock':clock,'rows':list(completed.values()),'summary':summarize(list(completed.values())),
              'reason':'Frozen capture was interrupted; original clock retained and capture is never silently restarted','official_test_pixels_read':0,'learning_blocked_by_fidelity':False,'finished_unix':time.time()}
            atomic_json(artifact/'diagnostic_result.json',report);return report
        if capture_record is None:
            save_checkpoint(folder,identity,completed,None,'FROZEN_TRAIN_CAPTURE')
            capture_job={'key':'FROZEN_TRAIN_CAPTURE','capture_only':True,'artifact':str(artifact),'profile':profile,'deadline':limit,'source_hashes':sources,'request_sha256':request_sha,'result_path':str(artifact/'capture_result.json')}
            capture_job_path=artifact/'capture_job.json';immutable(capture_job_path,capture_job)
            capture_row=_subprocess_case(root,capture_job_path,min(120.,limit-time.time()))
            if capture_row.get('status')!='CAPTURED':
                completed['FROZEN_TRAIN_CAPTURE']=capture_row;save_checkpoint(folder,identity,completed,None,None)
                report={'schema':SCHEMA,'status':'UNKNOWN','profile':profile,'request':request,'clock':clock,'rows':list(completed.values()),'summary':summarize(list(completed.values())),
                  'reason':'Frozen capture failed or reached hard120s bound; no capture retry or renewed clock','official_test_pixels_read':0,'learning_blocked_by_fidelity':False,'finished_unix':time.time()}
                atomic_json(artifact/'diagnostic_result.json',report);return report
            capture_record=capture_row['capture_record'];capture=torch.load(capture_record['path'],map_location='cpu',weights_only=False);completed.pop('FROZEN_TRAIN_CAPTURE',None)
            save_checkpoint(folder,identity,completed,capture_record,None)
        else:capture=torch.load(capture_record['path'],map_location='cpu',weights_only=False)
        layers=sorted(int(layer)for layer in capture['layers']);jobs=[(layer,rank,seed)for layer in layers for rank in request['ranks']for seed in request['basis_seeds']]
        for layer,rank,seed in jobs:
            key='l'+str(layer)+'-r'+str(rank)+'-s'+str(seed)
            if key in completed:continue
            if time.time()+3>=limit:
                completed[key]={'key':key,'layer':layer,'rank':rank,'feature_seed':seed,'status':'UNKNOWN_GLOBAL_DEADLINE'};save_checkpoint(folder,identity,completed,capture_record,None);continue
            job={'key':key,'profile':profile,'layer':layer,'rank':rank,'feature_seed':seed,'capture_record':capture_record,'source_hashes':sources,'request_sha256':request_sha,
              'deadline':limit,'result_path':str(artifact/'cases'/key/'result.json')}
            job_path=artifact/'cases'/key/'job.json';immutable(job_path,job);save_checkpoint(folder,identity,completed,capture_record,key)
            row=_subprocess_case(root,job_path,min(120.,limit-time.time()));row.update(layer=layer,rank=rank,feature_seed=seed)
            completed[key]=row;save_checkpoint(folder,identity,completed,capture_record,None);atomic_json(artifact/'diagnostic_progress.json',{'status':'RUNNING','completed_count':len(completed),'requested_cases':len(jobs),'rows':list(completed.values()),'clock':clock})
        rows=list(completed.values());complete=all(row.get('status')=='MEASURED'for row in rows)and len(rows)==len(jobs)
        report={'schema':SCHEMA,'status':'COMPLETE'if complete else'UNKNOWN','profile':profile,'request':request,'clock':clock,'capture_record':capture_record,'rows':rows,
          'summary':summarize(rows),'official_test_pixels_read':0,'learning_blocked_by_fidelity':False,'finished_unix':time.time(),
          'paths':{'registration':str(artifact/'diagnostic_registration.json'),'result':str(artifact/'diagnostic_result.json'),'checkpoint_manifest':str(folder/'manifest.json')}}
    except TimeoutError as error:
        report={'schema':SCHEMA,'status':'UNKNOWN','profile':profile,'request':request,'clock':clock,'rows':list(completed.values()),'summary':summarize(list(completed.values())),
          'reason':str(error),'official_test_pixels_read':0,'learning_blocked_by_fidelity':False,'finished_unix':time.time()}
    atomic_json(artifact/'diagnostic_result.json',report);return report


def run_diagnostic(root=ROOT,artifact=None,deadline=None,profile='full'):
    """Single-writer bounded stage. Full study launch owns its detached tmux."""
    if artifact is None:artifact=Path(root)/'artifacts/notebook_18/learned-kernel-001/diagnostic'
    artifact=Path(artifact);artifact.mkdir(parents=True,exist_ok=True)
    with (artifact/'diagnostic_writer.lock').open('a')as handle:
        try:fcntl.flock(handle,fcntl.LOCK_EX|fcntl.LOCK_NB)
        except BlockingIOError:raise RuntimeError('Another diagnostic writer already owns this run')
        try:return _run_diagnostic(root,artifact,deadline,profile)
        finally:fcntl.flock(handle,fcntl.LOCK_UN)


### Execute through the full study

These cells export reusable definitions only. Run the Notebook 18 front door to start the real diagnostic and learning study in detached tmux. `tests/18_Diagnostics_Validation.ipynb` independently validates the mathematics and continuation state without reading training or test images.